In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score
    )
from sklearn.utils.class_weight import compute_class_weight

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

In [ ]:
# Choose dataset here: mustafa | alex_binary | alex_binary_5050 | alex_012
DATASET_CHOICE = 'mustafa'

# For alex_012 only:
# 'strict' -> positive only if Diabetes_012 == 2
# 'merged' -> positive if Diabetes_012 > 0 (prediabetes + diabetes)
ALEX012_TARGET_MODE = 'strict'

DATASET_PATHS = {
    'mustafa': '/kaggle/input/datasets/iammustafatz/diabetes-prediction-dataset/diabetes_prediction_dataset.csv',
    'alex_012': '/kaggle/input/datasets/alexteboul/diabetes-health-indicators-dataset/diabetes_012_health_indicators_BRFSS2015.csv',
    'alex_binary_5050': '/kaggle/input/datasets/alexteboul/diabetes-health-indicators-dataset/diabetes_binary_5050split_health_indicators_BRFSS2015.csv',
    'alex_binary': '/kaggle/input/datasets/alexteboul/diabetes-health-indicators-dataset/diabetes_binary_health_indicators_BRFSS2015.csv'
}

if DATASET_CHOICE not in DATASET_PATHS:
    raise ValueError(f"Unknown DATASET_CHOICE={DATASET_CHOICE}. Choose one of: {list(DATASET_PATHS.keys())}")

if ALEX012_TARGET_MODE not in ['strict', 'merged']:
    raise ValueError("ALEX012_TARGET_MODE must be 'strict' or 'merged'")

raw_df = pd.read_csv(DATASET_PATHS[DATASET_CHOICE])
print(f"Dataset: {DATASET_CHOICE}")
if DATASET_CHOICE == 'alex_012':
    print(f"alex_012 target mode: {ALEX012_TARGET_MODE}")
print(raw_df.shape)
print(raw_df.dtypes.head(12))
print(raw_df.isnull().sum().head(12))

In [ ]:
# Build X, y depending on selected dataset
if DATASET_CHOICE == 'mustafa':
    df = raw_df.copy()

    # Keep only Female/Male because 'Other' is extremely rare
    df = df[df['gender'].isin(['Female', 'Male'])].reset_index(drop=True)

    # Encode categorical columns
    df['gender'] = df['gender'].map({'Female': 0, 'Male': 1}).astype(int)
    smoking_map = {
        'never': 0,
        'No Info': 1,
        'former': 2,
        'ever': 2,
        'not current': 2,
        'current': 3
    }
    df['smoking_history'] = df['smoking_history'].map(smoking_map).fillna(1).astype(int)

    y = df['diabetes'].astype(int).values
    X_df = df.drop(columns=['diabetes'])

elif DATASET_CHOICE == 'alex_binary':
    df = raw_df.copy()
    y = df['Diabetes_binary'].astype(int).values
    X_df = df.drop(columns=['Diabetes_binary'])

elif DATASET_CHOICE == 'alex_binary_5050':
    df = raw_df.copy()
    y = df['Diabetes_binary'].astype(int).values
    X_df = df.drop(columns=['Diabetes_binary'])

elif DATASET_CHOICE == 'alex_012':
    df = raw_df.copy()
    if ALEX012_TARGET_MODE == 'strict':
        y = (df['Diabetes_012'] == 2).astype(int).values
    else:
        y = (df['Diabetes_012'] > 0).astype(int).values
    X_df = df.drop(columns=['Diabetes_012'])

# Safety: encode any remaining object columns
object_cols = X_df.select_dtypes(include=['object']).columns
if len(object_cols) > 0:
    X_df = pd.get_dummies(X_df, columns=object_cols, drop_first=True)

# Safety: numeric cast + median imputation
X_df = X_df.apply(pd.to_numeric, errors='coerce')
X_df = X_df.fillna(X_df.median(numeric_only=True))
X = X_df.values.astype(np.float32)

print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"Class balance -> No diabetes: {(y==0).sum()}, Diabetes: {(y==1).sum()}")
print(f"Positive rate: {(y==1).mean():.4f}")

In [ ]:
# Split into train/validation/test (64/16/20)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f"Train size: {len(y_train)}, Val size: {len(y_val)}, Test size: {len(y_test)}")

In [ ]:
class DiabetesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_dataset = DiabetesDataset(X_train, y_train)
val_dataset   = DiabetesDataset(X_val, y_val)
test_dataset  = DiabetesDataset(X_test, y_test)

# For alex_012, avoid double imbalance correction: no weighted sampler
if DATASET_CHOICE == 'alex_012':
    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
else:
    class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    sample_weights = [class_weights[int(l)] for l in y_train]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_dataset, batch_size=256, sampler=sampler)

val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [ ]:
class DiabetesNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
            # No Sigmoid here — BCEWithLogitsLoss handles it
        )

    def forward(self, x):
        return self.net(x)

model = DiabetesNet(input_dim=X_train.shape[1]).to(device)

In [ ]:
# Loss setup: for alex_012 use neutral weight because sampler is disabled
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
ratio = neg_count / max(pos_count, 1)

if DATASET_CHOICE == 'alex_012':
    pos_weight_value = 1.0
else:
    pos_weight_value = float(np.clip(ratio, 1.0, 8.0))

pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

print(f"Using pos_weight={pos_weight_value:.3f} (neg={neg_count}, pos={pos_count})")

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, all_labels, all_probs = 0, [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            logits = model(X_b)
            probs = torch.sigmoid(logits)
            total_loss += criterion(logits, y_b).item()
            all_probs.extend(probs.cpu().numpy().ravel())
            all_labels.extend(y_b.cpu().numpy().ravel())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels).astype(int)
    auc = roc_auc_score(all_labels, all_probs)
    pr_auc = average_precision_score(all_labels, all_probs)
    return total_loss / len(loader), all_labels, all_probs, auc, pr_auc

best_val_loss = float('inf')
patience_counter = 0
PATIENCE = 10

for epoch in range(100):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_labels, val_probs, val_auc, val_pr_auc = eval_epoch(model, val_loader, criterion)

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stop at epoch {epoch+1}")
            break

    if (epoch + 1) % 5 == 0:
        print(
            f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val ROC-AUC: {val_auc:.4f} | Val PR-AUC: {val_pr_auc:.4f}"
        )

In [ ]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

# Threshold selection policy (validation set only)
# Options: 'f1', 'precision_at_min_recall', 'recall_at_min_precision'
THRESHOLD_MODE = 'precision_at_min_recall'
MIN_RECALL_TARGET = 0.55
MIN_PRECISION_TARGET = 0.45

def select_threshold(labels, probs, mode='f1', min_recall=0.55, min_precision=0.45):
    thresholds = np.linspace(0.05, 0.95, 91)
    rows = []

    for thr in thresholds:
        preds = (probs >= thr).astype(int)
        p = precision_score(labels, preds, zero_division=0)
        r = recall_score(labels, preds, zero_division=0)
        f1 = f1_score(labels, preds, zero_division=0)
        rows.append({'thr': thr, 'precision': p, 'recall': r, 'f1': f1})

    metric_df = pd.DataFrame(rows)

    if mode == 'precision_at_min_recall':
        candidates = metric_df[metric_df['recall'] >= min_recall]
        if len(candidates) > 0:
            best = candidates.sort_values(['precision', 'f1', 'thr'], ascending=[False, False, False]).iloc[0]
        else:
            best = metric_df.sort_values(['recall', 'f1'], ascending=[False, False]).iloc[0]

    elif mode == 'recall_at_min_precision':
        candidates = metric_df[metric_df['precision'] >= min_precision]
        if len(candidates) > 0:
            best = candidates.sort_values(['recall', 'f1'], ascending=[False, False]).iloc[0]
        else:
            best = metric_df.sort_values(['precision', 'f1'], ascending=[False, False]).iloc[0]

    else:
        best = metric_df.sort_values('f1', ascending=False).iloc[0]

    return float(best['thr']), metric_df

# 1) Tune threshold on validation set
_, val_labels, val_probs, val_roc_auc, val_pr_auc = eval_epoch(model, val_loader, criterion)
best_thr, threshold_grid = select_threshold(
    val_labels, val_probs, mode=THRESHOLD_MODE, min_recall=MIN_RECALL_TARGET, min_precision=MIN_PRECISION_TARGET
    )

# 2) Evaluate once on untouched test set
_, test_labels, test_probs, test_roc_auc, test_pr_auc = eval_epoch(model, test_loader, criterion)
test_preds = (test_probs >= best_thr).astype(int)

test_precision = precision_score(test_labels, test_preds, zero_division=0)
test_recall = recall_score(test_labels, test_preds, zero_division=0)
test_f1 = f1_score(test_labels, test_preds, zero_division=0)

print(f"\nValidation ROC-AUC: {val_roc_auc:.4f}")
print(f"Validation PR-AUC : {val_pr_auc:.4f}")
if THRESHOLD_MODE == 'precision_at_min_recall':
    print(f"Threshold policy: maximize precision with recall >= {MIN_RECALL_TARGET:.2f}")
elif THRESHOLD_MODE == 'recall_at_min_precision':
    print(f"Threshold policy: maximize recall with precision >= {MIN_PRECISION_TARGET:.2f}")
else:
    print("Threshold policy: maximize F1")
print(f"Chosen threshold from validation: {best_thr:.2f}\n")

print(f"Test ROC-AUC: {test_roc_auc:.4f}")
print(f"Test PR-AUC : {test_pr_auc:.4f}\n")
print(classification_report(test_labels, test_preds, target_names=['No Diabetes', 'Diabetes'], zero_division=0))

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['No Diabetes', 'Diabetes'],
    yticklabels=['No Diabetes', 'Diabetes']
    )
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Test Confusion Matrix @ threshold={best_thr:.2f}')
plt.tight_layout()
plt.show()

# Precision-Recall curve (test)
precisions, recalls, _ = precision_recall_curve(test_labels, test_probs)
plt.figure(figsize=(6, 5))
plt.plot(recalls, precisions, color='darkorange', linewidth=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Test Precision-Recall Curve')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Persistent run log in notebook session
if 'EXPERIMENT_LOG' not in globals():
    EXPERIMENT_LOG = []

run_row = {
    'dataset': DATASET_CHOICE,
    'alex012_mode': ALEX012_TARGET_MODE if DATASET_CHOICE == 'alex_012' else '-',
    'threshold_mode': THRESHOLD_MODE,
    'chosen_threshold': round(float(best_thr), 4),
    'val_roc_auc': round(float(val_roc_auc), 4),
    'val_pr_auc': round(float(val_pr_auc), 4),
    'test_roc_auc': round(float(test_roc_auc), 4),
    'test_pr_auc': round(float(test_pr_auc), 4),
    'test_precision_diabetes': round(float(test_precision), 4),
    'test_recall_diabetes': round(float(test_recall), 4),
    'test_f1_diabetes': round(float(test_f1), 4)
}
EXPERIMENT_LOG.append(run_row)

results_df = pd.DataFrame(EXPERIMENT_LOG).drop_duplicates()
results_df = results_df.sort_values(['dataset', 'test_pr_auc', 'test_f1_diabetes'], ascending=[True, False, False])
print("\nExperiment comparison table:")
display(results_df)

# Optional: save to a file in Kaggle working directory
results_df.to_csv('/kaggle/working/experiment_comparison.csv', index=False)
print("Saved: /kaggle/working/experiment_comparison.csv")

In [ ]:
Experiment comparison table:
dataset	alex012_mode	threshold_mode	chosen_threshold	val_roc_auc	val_pr_auc	test_roc_auc	test_pr_auc	test_precision_diabetes	test_recall_diabetes	test_f1_diabetes
4	alex_012	merged	precision_at_min_recall	0.31	0.8272	0.4621	0.8250	0.4620	0.4359	0.5537	0.4878
3	alex_012	strict	precision_at_min_recall	0.26	0.8287	0.4342	0.8266	0.4185	0.3989	0.5526	0.4633
1	alex_binary	-	precision_at_min_recall	0.92	0.8279	0.4296	0.8255	0.4159	0.3813	0.5907	0.4635
2	alex_binary_5050	-	precision_at_min_recall	0.70	0.8321	0.8098	0.8302	0.8050	0.8033	0.5586	0.6590
0	mustafa	-	precision_at_min_recall	0.95	0.9774	0.8770	0.9767	0.8753	0.7339	0.7853	0.7587